# 3D arm — CT nodule malignancy (Colab GPU)

Runs the **3D ResNet-10** experiments with the *identical* protocol used for the
2D and 2.5D arms: same HU window, same splits, same label policy, same loss,
optimiser, augmentation, early stopping and metrics. Only the input
representation and the encoder dimensionality change.

**Why Colab:** 3D convolutions take ~14 min/epoch on an Apple M2 (≈2 weeks for
the full set). A free T4 does it in a fraction of that.

**Before running:** set the runtime to a GPU — *Runtime → Change runtime type → T4 GPU*.


## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 2. Clone the code

The repository is private, so use a GitHub personal access token with `repo`
scope (github.com → Settings → Developer settings → Tokens). It is only stored
in this runtime.

In [ ]:
from getpass import getpass
TOKEN = getpass("GitHub token: ")
USER  = "naazzarov"
REPO  = "CT-2D-2.5D-3D-DeepLearning"
!git clone -q https://{TOKEN}@github.com/{USER}/{REPO}.git /content/repo
%cd /content/repo
!ls src/3d

In [ ]:
!pip -q install scikit-learn pandas matplotlib seaborn
print("deps ready")

## 3. Mount the data

Upload two things to your Google Drive beforehand:

1. `volumes_fp16.npy` + `volumes_fp16_index.json` — produced locally with
   `python scripts/export_volumes_fp16.py` (~8.5 GB, lossless int16).
2. The `metadata/` folder from `final_team_dataset_v2_3class`, and the
   `binary_sensitivity_team_package/` folder.

Adjust the paths below to match where you put them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = "/content/drive/MyDrive/ct_project"     # <-- edit if needed

os.environ["CT_DATA_ROOT"]    = f"{DRIVE}/final_team_dataset_v2_3class"
os.environ["CT_BINARY_ROOT"]  = f"{DRIVE}/binary_sensitivity_team_package"
os.environ["CT_VOLUME_PACK"]  = f"{DRIVE}/volumes_fp16.npy"

for k in ("CT_DATA_ROOT","CT_BINARY_ROOT","CT_VOLUME_PACK"):
    print(k, "->", os.environ[k], "| exists:", os.path.exists(os.environ[k]))

## 4. Verify the data before training

This asserts zero patient/nodule leakage, the agreed binary counts, and that
volumes load at the expected shape. It **raises** rather than warns.

In [ ]:
import sys; sys.path.insert(0, "/content/repo")
from src.common.data import load_splits, load_ssl_pool, assert_no_leakage, verify_binary_counts
from src.common.data3d import VolumeDataset, volume_train_statistics

s3 = load_splits("3class"); sb = load_splits("binary")
assert_no_leakage(s3, load_ssl_pool())
verify_binary_counts(sb)
assert_no_leakage(sb, load_ssl_pool(), reference=s3)
print("3-class:", {k: len(v) for k, v in s3.items()})
print("binary :", {k: len(v) for k, v in sb.items()})

mean, std = volume_train_statistics(s3["train"], limit=100)
print("volume intensity mean %.4f std %.4f" % (mean, std))
ds = VolumeDataset(s3["val"], mean, std)
x, y = ds[0]; print("sample shape", tuple(x.shape), "label", y)

## 5. Speed check

Confirms the GPU is actually being used before you commit to long runs.

In [ ]:
import time, torch, sys
sys.path.insert(0, "/content/repo/src/3d")
from model import ResNet10Classifier, n_params
dev = "cuda"
m = ResNet10Classifier().to(dev)
opt = torch.optim.AdamW(m.parameters(), lr=1e-3); crit = torch.nn.CrossEntropyLoss()
x = torch.randn(16,1,104,72,80, device=dev); y = torch.randint(0,3,(16,), device=dev)
for _ in range(3): opt.zero_grad(); crit(m(x),y).backward(); opt.step()
torch.cuda.synchronize(); t=time.perf_counter()
for _ in range(10): opt.zero_grad(); crit(m(x),y).backward(); opt.step()
torch.cuda.synchronize()
per=(time.perf_counter()-t)/10
print(f"{per:.3f} s/step  ->  {1876/16*per/60:.1f} min/epoch   (M2 was 13.5)")
print("params %.2fM" % (n_params(m)/1e6))

## 6. Model A — 3-class supervised, no augmentation (3 seeds)

In [ ]:
!for s in 0 1 2; do python src/3d/train_supervised.py --task 3class \
    --init random --aug none --tag model_a_noaug --seed $s \
    --epochs 80 --patience 20 --batch-size 16 --eval-test 2>&1 | grep -vE "^ep +[0-9]+ \|"; done

## 7. Model C — 3-class supervised, mild augmentation (3 seeds)

In [ ]:
!for s in 0 1 2; do python src/3d/train_supervised.py --task 3class \
    --init random --aug mild --tag model_c_mild --seed $s \
    --epochs 80 --patience 20 --batch-size 16 --eval-test 2>&1 | grep -vE "^ep +[0-9]+ \|"; done

## 8. Binary sensitivity — mild augmentation (3 seeds)

In [ ]:
!for s in 0 1 2; do python src/3d/train_supervised.py --task binary \
    --init random --aug mild --tag binary_mild --seed $s \
    --epochs 80 --patience 20 --batch-size 16 --eval-test 2>&1 | grep -vE "^ep +[0-9]+ \|"; done

## 9. MoCo v2 pretraining (the long one)

5133 training-patient volumes, no labels. Start with 100 epochs; raise it if
the runtime holds. Save the checkpoint to Drive so a disconnect doesn't lose it.

In [ ]:
!python src/3d/moco_pretrain.py --epochs 100 --batch-size 32 --workers 2
!mkdir -p "{DRIVE}/ckpt" && cp results/3d/moco/moco_encoder.pt "{DRIVE}/ckpt/" && echo "checkpoint backed up to Drive"

## 10. Model B — MoCo fine-tuning (3 seeds)

In [ ]:
!for s in 0 1 2; do python src/3d/train_supervised.py --task 3class \
    --init moco --moco-ckpt results/3d/moco/moco_encoder.pt --aug none \
    --tag model_b_noaug --seed $s \
    --epochs 80 --patience 20 --batch-size 16 --eval-test 2>&1 | grep -vE "^ep +[0-9]+ \|"; done

## 11. Save results back to the repo

Results are small JSON/PNG files. This commits them so the 3D numbers land
alongside the 2D and 2.5D ones.

In [ ]:
!cp -r results/3d "{DRIVE}/results_3d_backup" && echo "backed up to Drive"
!git -c user.email="191228853+naazzarov@users.noreply.github.com" -c user.name="Bahodir" add -A results/3d
!git -c user.email="191228853+naazzarov@users.noreply.github.com" -c user.name="Bahodir" commit -q -m "3D results from Colab GPU run" && git push && echo "pushed" 